## Problem Statement

### Business Context

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

**Common Questions to Answer**

**1. Diagnostic Assistance**: "What are the common symptoms and treatments for pulmonary embolism?"

**2. Drug Information**: "Can you provide the trade names of medications used for treating hypertension?"

**3. Treatment Plans**: "What are the first-line options and alternatives for managing rheumatoid arthritis?"

**4. Specialty Knowledge**: "What are the diagnostic steps for suspected endocrine disorders?"

**5. Critical Care Protocols**: "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to **understand** issues like information overload, **apply** AI techniques to streamline decision-making, **analyze** its impact on diagnostics and patient outcomes, **evaluate** its potential to standardize care practices, and **create** a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The **Merck Manuals** are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 67.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 336.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 300.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.6/16.6 MB 222.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.6/44.6 kB 246.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-python-headless 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 2.3.3 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.3 which is incompatible.
cupy-cuda12x 13.3.0 requires numpy<2.3,>=1.22, but you have n

In [ ]:
!python -m pip install -q --upgrade pip setuptools wheel

# Core AI + RAG + Experiment Tracking dependencies
!python -m pip install -q \
  "huggingface_hub>=0.25.2" \
  "pandas>=2.2.2" \
  "tiktoken>=0.7.0" \
  "pymupdf>=1.24.10" \
  "langchain>=0.2.14" \
  "langchain-community>=0.2.10" \
  "chromadb>=0.5.3" \
  "sentence-transformers>=3.0.1" \
  "numpy>=1.26.4" \
  "openai>=1.45.0" \
  "mlflow>=2.16.2" \
  "langchain-openai>=0.1.7" \
  "langchain-chroma>=0.1.1" \
  "scipy>=1.11.4" \
  "scikit-learn>=1.4.2" \


#The above mentioned Libraries to be installed. Always check the compatible versions that runs with  LLaMA C++ Python bindings. This package lets Python talk to Meta’s LLaMA model (and other GGUF-format models).

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 49.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.3 which is incompatible.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.
tensorflow 2.19.0 requires numpy<2.2.0,>=1.26.0, but you have numpy 2.3.3 which is incompatible.


# **Importing the Libraries**

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd
import chromadb, mlflow, openai, langchain

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

#Libraries for downloading and loading the llm
from huggingface_hub import hf_hub_download
from llama_cpp import Llama

# **## Question Answering using LLM**

#### Downloading and Loading the model

In [ ]:
#Hugging Face model repository
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"

#Actual model file inside that repo
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

In [ ]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,    # Repository ID
    filename=model_basename        # Specific model file to download
)

print("Model downloaded to:", model_path)

mistral-7b-instruct-v0.2.Q6_K.gguf:   0%|          | 0.00/5.94G [00:00<?, ?B/s]

Model downloaded to: /root/.cache/huggingface/hub/models--TheBloke--Mistral-7B-Instruct-v0.2-GGUF/snapshots/3a6fbf4a41a1d52e415a4958cde6856d34b2db93/mistral-7b-instruct-v0.2.Q6_K.gguf


**Load the model with llama-cpp-python**

In [ ]:
llm = Llama(
    model_path=model_path,   # downloaded .gguf file
    n_ctx=4096,              # context window (max tokens per prompt)
    n_gpu_layers=38,         # number of layers to run on GPU (T4)
    n_batch=512              # batch size for generation

)

print("LLaMA / Mistral model loaded successfully.")

LLaMA / Mistral model loaded successfully.


AVX = 1 | AVX2 = 1 | AVX512 = 0 | AVX512_VBMI = 0 | AVX512_VNNI = 0 | FMA = 1 | NEON = 0 | ARM_FMA = 0 | F16C = 1 | FP16_VA = 0 | WASM_SIMD = 0 | BLAS = 1 | SSE3 = 1 | SSSE3 = 1 | VSX = 0 | 


# **Response**

Below Function (response) will run the Mistral model on GPU attached for this session. The parameters  (temperarture, top_p, top_k) will combine to shape creativity vs reliability of the answer generated.

In [ ]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

**Check for sample response with a prompt**

In [ ]:
response("What treatment options are available for managing hypertension?")

'\n\nHypertension, or high blood pressure, is a common condition that can increase the risk of various health problems such as heart disease, stroke, and kidney damage. The good news is that there are several effective treatment options available to help manage hypertension and reduce the risk of complications. Here are some of the most commonly used treatments:\n\n1. Lifestyle modifications: Making lifestyle changes is often the first line of defense against hypertension. This may include eating a healthy diet rich in fruits, vegetables, whole grains, and lean proteins; limiting sodium intake; getting regular physical activity'

## Question Answering using LLM with Prompt Engineering

## Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
response(user_input)

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. Resusc'

**The above response is generated as expected. Upon on observing the response, it is constrained to max of 128 tokens, where it stopped processing at (2 point)'Resusc'**

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
response(user_input_2)

Llama.generate: prefix-match hit


'\n\nAppendicitis is a medical condition characterized by inflammation of the appendix, a small tube-shaped organ located in the lower right side of the abdomen. The symptoms of appendicitis can vary from person to person, but some common signs include:\n\n1. Abdominal pain: The pain may start as a mild discomfort around the navel or in the lower right abdomen, which then gradually moves to the right lower quadrant and becomes more severe over time. The pain may be constant or intermittent and is often worsened by movement, coughing, or deep breathing'

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
response(user_input_3)

Llama.generate: prefix-match hit


"\n\nSudden patchy hair loss, also known as alopecia areata, is a common autoimmune disorder that affects the hair follicles. It can result in round or oval bald patches on the scalp, but it can also occur on other parts of the body such as the beard area, eyebrows, or eyelashes.\n\nThe exact cause of alopecia areata is not known, but it's believed to be related to a problem with the immune system. Some possible triggers for this condition include stress, genetics, viral infections, and certain medications."

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4)

Llama.generate: prefix-match hit


"\n\nA person who has sustained a physical injury to brain tissue, also known as a traumatic brain injury (TBI), may require various treatments depending on the severity and location of the injury. Here are some common treatments recommended for TBIs:\n\n1. Emergency care: The first priority is to ensure the person's airway is clear, they are breathing, and their heart is beating normally. In severe cases, emergency surgery may be required to remove hematomas or other obstructions.\n2. Medications: Depending on the symptoms, medications may be prescribed to manage conditions such as"

**Based on previous reponse, modyfying the max_tokens value from 128 to 139, in the response, we have got the proper sentence that makes sense ending with fullstop '.'**

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
response(user_input_4, max_tokens=139)

Llama.generate: prefix-match hit


'\n\nA person who has sustained a physical injury to the brain tissue may require various treatments depending on the severity and location of the injury. Here are some common treatments that may be recommended:\n\n1. Emergency care: In case of a traumatic brain injury (TBI), it is essential to seek emergency medical attention as soon as possible. The primary goal of emergency care is to prevent further damage to the brain, stabilize vital signs, and manage any life-threatening conditions.\n2. Medications: Depending on the symptoms, healthcare professionals may prescribe medications to manage various conditions such as pain, swelling, seizures, or infections.'

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
response(user_input_5,max_tokens=144)

Llama.generate: prefix-match hit


"\n\nFirst and foremost, if you suspect that someone has fractured their leg while hiking, it's essential to ensure their safety and prevent further injury. Here are some necessary precautions:\n\n1. Keep the person calm and still: Encourage them to remain as still as possible to minimize pain and prevent worsening the injury.\n2. Assess the situation: Check for any signs of shock, such as pale skin, rapid heartbeat, or shallow breathing. If you notice these symptoms, seek medical help immediately.\n3. Immobilize the leg: Use a splint, sling, or other available materials to immobilize the leg and prevent movement."

## Data Preparation for RAG

### Loading the Data

Mount the Drive to access the medical_diagnosis_manual.pdf

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Define the path for Document (medical_diagnosis_manual)

In [ ]:
manual_pdf_path = "/content/drive/MyDrive/Great_Learning_AI_ML_Projects/Natural_Language_Processing_Generative_AI_LLM_RAG/medical_diagnosis_manual.pdf"

PyMuPDFLoader is a document loader specifically designed for working with PDF files within the LangChain framework. It leverages the high-performance PyMuPDF library, a Python binding for the MuPDF PDF processing library, to extract content from PDFs.

In [ ]:
pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [ ]:
manual = pdf_loader.load()

# **Data Overview**

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

Page Number : 1
pramodkumar.sola@gmail.com
OJQB9PWIF5
 for personal use by pramodkumar.sola@
shing the contents in part or full is liable
Page Number : 2
pramodkumar.sola@gmail.com
OJQB9PWIF5
This file is meant for personal use by pramodkumar.sola@gmail.com only.
Sharing or publishing the contents in part or full is liable for legal action.
Page Number : 3
Table of Contents
1
Front    ................................................................................................................................................................................................................
1
Cover    .......................................................................................................................................................................................................
2
Front Matter    ..............................................................................................................................................................................

#### Checking the number of pages

In [ ]:
len(manual)

print(f"Number of pages : {len(manual)}")

Number of pages : 4114


### Data Chunking

RecursiveCharacterTextSplitter a text-splitting tool, commonly used in natural language processing, that breaks large documents into smaller, manageable chunks.

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=1000, #code to define the chunk size
    chunk_overlap=150 # code to define the chunk overlap
)

In [ ]:
document_chunks = pdf_loader.load_and_split(text_splitter)

In [ ]:
len(document_chunks)
print(f"Number of chunks : {len(document_chunks)}")

Number of chunks : 4703


**Check some sample content after chunk**

In [ ]:
document_chunks[0].page_content

'pramodkumar.sola@gmail.com\nOJQB9PWIF5\n for personal use by pramodkumar.sola@\nshing the contents in part or full is liable'

In [ ]:
document_chunks[2].page_content

"Table of Contents\n1\nFront    ................................................................................................................................................................................................................\n1\nCover    .......................................................................................................................................................................................................\n2\nFront Matter    ...........................................................................................................................................................................................\n53\n1 - Nutritional Disorders    ...............................................................................................................................................................\n53\nChapter 1. Nutrition: General Considerations    ...........................................................................................

In [ ]:
document_chunks[3].page_content

'491\nChapter 44. Foot & Ankle Disorders    .....................................................................................................................................\n502\nChapter 45. Tumors of Bones & Joints    ...............................................................................................................................\n510\n5 - Ear, Nose, Throat & Dental Disorders    ..................................................................................................................\n510\nChapter 46. Approach to the Patient With Ear Problems    ...........................................................................................\n523\nChapter 47. Hearing Loss    .........................................................................................................................................................\n535\nChapter 48. Inner Ear Disorders    ...................................................................................................

In [ ]:
document_chunks[-1].page_content

"Z\nZafirlukast 1879\nZalcitabine 1451\nin children 2854\nZaleplon 1709\nZanamivir 1407\nin influenza 1407, 1929\nZAP-70 (zeta-associated protein 70) deficiency 1092, 1108\nZavanelli maneuver 2680\nZellweger syndrome 2383, 3023\nZenker's diverticulum 125\nZidovudine 1451, 1453\nin children 2854\nZileuton 1881\nin asthma 1880\nZinc 49, 55, 3431-3432\nin common cold 1405\ndeficiency of 11, 49, 55\nin dermatophytoses 705\npoisoning with 3328, 3353\nrecommended dietary allowances for 50\nreference values for 3499\ntoxicity of 49, 55\ncopper deficiency and 49\nin Wilson's disease 52\nZinc oxide 2233\ngelatin formulation of 646, 672\nZinc pyrithione 647\nZinc shakes 55\nZipper injury 3239, 3240\nZiprasidone\nin agitation 1492\nin bipolar disorder 3059\npoisoning with 3347\nin schizophrenia 1566\nZoledronate 359, 361, 848\nZollinger-Ellison syndrome 95, 199, 200-201, 910\nmastocytosis vs 1125\nMenetrier's disease vs 132\npeptic ulcer disease vs 134\nZolmitriptan 1721\nZolpidem 1709, 3103\nZon

# **Embedding**

 For reference model comparisions visit  
(https://huggingface.co/sentence-transformers)

In [ ]:
embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")

/tmp/ipython-input-3102610685.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = SentenceTransformerEmbeddings(model_name="thenlper/gte-large")


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/619 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/342 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

**Below shows the Semantic vectors**

In [ ]:
print(embedding_2)

[-0.009725572541356087, 0.0004775130655616522, 0.016911551356315613, -0.0009186926763504744, -0.0018662909278646111, -0.01232975348830223, 0.003706393763422966, 0.014911845326423645, 0.023290881887078285, 0.020453890785574913, 0.010174417868256569, -8.041466207941994e-05, 0.022362979128956795, -0.03420141339302063, -0.015580808743834496, -0.0015397097449749708, -0.008691388182342052, -0.05539773032069206, 0.0007035742164589465, 0.0009258303907699883, -0.011358385905623436, 0.006325399968773127, -0.053631313145160675, -0.038284894078969955, 0.006373541429638863, 0.028255309909582138, 0.017489375546574593, -0.01163148321211338, 0.07090899348258972, 0.032021306455135345, -0.013494466431438923, -0.025929097086191177, 0.025900671258568764, -0.04591570794582367, -0.0015832888893783092, -0.009393211454153061, 0.057287540286779404, -0.01915176585316658, -0.009869030676782131, -0.030193498358130455, 0.014128674753010273, -0.015488401986658573, 0.044975876808166504, -0.04775606468319893, -0.0285

In [ ]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Dimension of the embedding vector  1024


True

In [ ]:
embedding_1,embedding_2

([-0.026187170296907425,
  -0.006192934699356556,
  0.00690766517072916,
  -0.013215618208050728,
  -0.0020496989600360394,
  -0.021427173167467117,
  0.0037614458706229925,
  0.012813618406653404,
  0.018012186512351036,
  0.011135362088680267,
  0.015805598348379135,
  0.006946881767362356,
  0.020669085904955864,
  -0.035063356161117554,
  -0.010018471628427505,
  -0.00017761792696546763,
  -0.011952176690101624,
  -0.0531720407307148,
  -0.0017641111044213176,
  0.006629332434386015,
  0.002226178999990225,
  0.0065945815294981,
  -0.06247532367706299,
  -0.03622320294380188,
  0.004393283743411303,
  0.020820315927267075,
  0.017670106142759323,
  -0.0003142147324979305,
  0.0720522329211235,
  0.02915252186357975,
  -0.02084224857389927,
  -0.009313223883509636,
  0.022168617695569992,
  -0.037334099411964417,
  -0.005411284975707531,
  0.00784269254654646,
  0.05414879322052002,
  -0.02199687995016575,
  -0.016822384670376778,
  -0.02541324682533741,
  0.021155251190066338,
  -0

### Vector Database

In [ ]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

Chroma.from_documents is a method used in the LangChain framework to initialize a Chroma vector store directly from a list of Document objects.

In [ ]:
vectorstore = Chroma.from_documents(document_chunks,embedding_model,
persist_directory=out_dir)

The folder on disk where your Chroma database is saved (e.g. 'medical_db'). It contains index files and vector data.

In [ ]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

/tmp/ipython-input-2756559696.py:1: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)


**Used the BertModel architecture with word_embedding_dimension = 1024, where ('pooling_mode_mean_tokens': True) reduing the token level vectors to single sentence level vector.**

In [ ]:
vectorstore.embeddings

HuggingFaceEmbeddings(client=SentenceTransformer(
  (0): Transformer({'max_seq_length': 512, 'do_lower_case': False, 'architecture': 'BertModel'})
  (1): Pooling({'word_embedding_dimension': 1024, 'pooling_mode_cls_token': False, 'pooling_mode_mean_tokens': True, 'pooling_mode_max_tokens': False, 'pooling_mode_mean_sqrt_len_tokens': False, 'pooling_mode_weightedmean_tokens': False, 'pooling_mode_lasttoken': False, 'include_prompt': True})
  (2): Normalize()
), model_name='thenlper/gte-large', cache_folder=None, model_kwargs={}, encode_kwargs={}, multi_process=False, show_progress=False)

**Testing sample similarity search**

In [ ]:
vectorstore.similarity_search("What are the common symptoms of appendicitis and how is it treated?",k=3)

[Document(metadata={'creator': 'Atop CHM to PDF Converter', 'author': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'source': '/content/drive/MyDrive/Great_Learning_AI_ML_Projects/Natural_Language_Processing_Generative_AI_LLM_RAG/medical_diagnosis_manual.pdf', 'creationDate': 'D:20120615054440Z', 'file_path': '/content/drive/MyDrive/Great_Learning_AI_ML_Projects/Natural_Language_Processing_Generative_AI_LLM_RAG/medical_diagnosis_manual.pdf', 'keywords': '', 'creationdate': '2012-06-15T05:44:40+00:00', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'modDate': 'D:20251011220107Z', 'moddate': '2025-10-11T22:01:07+00:00', 'subject': '', 'total_pages': 4114, 'format': 'PDF 1.7', 'page': 174, 'trapped': ''}, page_content="• Surgical removal\n• IV fluids and antibiotics\nTreatment of acute appendicitis is open or laparoscopic appendectomy; because treatment delay\nincreases mortality, a negative appendectomy rate of 15% is considered acceptable. The surgeo

### Retriever

The Chroma database that holds all the embedded document chunks (like Merck manual sections).

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k':3}
)

**Pass Sample prompt to check what relevant data is retrieved**

Using embedding model to embed the query, compares it with stored vectors using cosine similarity, and returns the top-k most similar ones.

In [ ]:
rel_docs = retriever.get_relevant_documents("What is the protocol for managing sepsis in a critical care unit?")
rel_docs

/tmp/ipython-input-1655728131.py:1: LangChainDeprecationWarning: The method `BaseRetriever.get_relevant_documents` was deprecated in langchain-core 0.1.46 and will be removed in 1.0. Use :meth:`~invoke` instead.
  rel_docs = retriever.get_relevant_documents("What is the protocol for managing sepsis in a critical care unit?")


[Document(metadata={'creationDate': 'D:20120615054440Z', 'total_pages': 4114, 'modDate': 'D:20251011220107Z', 'page': 2400, 'trapped': '', 'format': 'PDF 1.7', 'creator': 'Atop CHM to PDF Converter', 'file_path': '/content/drive/MyDrive/Great_Learning_AI_ML_Projects/Natural_Language_Processing_Generative_AI_LLM_RAG/medical_diagnosis_manual.pdf', 'creationdate': '2012-06-15T05:44:40+00:00', 'subject': '', 'moddate': '2025-10-11T22:01:07+00:00', 'source': '/content/drive/MyDrive/Great_Learning_AI_ML_Projects/Natural_Language_Processing_Generative_AI_LLM_RAG/medical_diagnosis_manual.pdf', 'producer': 'pdf-lib (https://github.com/Hopding/pdf-lib)', 'author': '', 'title': 'The Merck Manual of Diagnosis & Therapy, 19th Edition', 'keywords': ''}, page_content="16 - Critical Care Medicine\nChapter 222. Approach to the Critically Ill Patient\nIntroduction\nCritical care medicine specializes in caring for the most seriously ill patients. These patients are best\ntreated in an ICU staffed by expe

We can observe that the three relevant chunks above contain the answer to the query.
If we increase the k value, there is a chance that we might find the answer in even more chunks.
This is a hyperparameter that we need to tune to get the best context.

**The Below response is somewhat generic and is solely based on the data the model was trained on, rather than the medical manual.**

In [ ]:
model_output = llm(
      "What is the protocol for managing sepsis in a critical care unit?",
      max_tokens=500, # Pass the maximum number of tokens. The number 500 given may not ended the sentence properly with a fullstop '.'
      temperature=0.0, # pass the temperature value. Lower = Factual, HIgher = Creative
    )

model_output['choices'][0]['text']

Llama.generate: prefix-match hit


'\n\nSepsis is a life-threatening condition that can arise from an infection, and it requires prompt recognition and aggressive management in a critical care unit. The following are general steps for managing sepsis in a critical care unit:\n\n1. Early recognition: Recognize the signs and symptoms of sepsis early and initiate treatment as soon as possible. Sepsis can present with various clinical features, including fever or hypothermia, tachycardia or bradycardia, altered mental status, respiratory distress, and lactic acidosis.\n2. Source control: Identify and address the source of infection as quickly as possible. This may involve surgical intervention, such as drainage of an abscess or debridement of necrotic tissue.\n3. Fluid resuscitation: Administer intravenous fluids to maintain adequate blood pressure and organ perfusion. The goal is to achieve a mean arterial pressure (MAP) of at least 65 mmHg and a central venous oxygen saturation (ScvO2) of greater than 70%.\n4. Vasopressor

### System and User Prompt Template

Prompts guide the model to generate accurate responses. Here, we define two parts:

1. The system message describing the assistant's role.
2. A user message template including context and the question.

**This tells the LLM who it is and how it should behave and respomd.**

In [ ]:
qna_system_message = (
    "You are a knowledgeable and reliable medical assistant. "
    "Use only trusted medical information (like Merck Manuals) to answer. "
    "Provide clear, evidence-based explanations suitable for healthcare professionals. "
    "If the answer is not found in the given context, say you don’t have enough information."
)

**This defines how the user query and context are passed to the model.**

In [ ]:
qna_user_message_template = (
    "You are given the following medical context:\n\n"
    "{context}\n\n"
    "Based on the above, answer the following question accurately and concisely:\n"
    "Question: {question}\n\n"
    "Answer:"
)

# **Response Function**

In [ ]:
def generate_rag_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=k)
    context_list = [d.page_content for d in relevant_document_chunks]

    # Combine document chunks into a single context
    context_for_query = ". ".join(context_list)

    user_message = qna_user_message_template.replace('{context}', context_for_query)
    user_message = user_message.replace('{question}', user_input)

    prompt = qna_system_message + '\n' + user_message

    # Generate the response
    try:
        response = llm(
                  prompt=prompt,
                  max_tokens=max_tokens,
                  temperature=temperature,
                  top_p=top_p,
                  top_k=top_k
                  )

        # Extract and print the model's response
        response = response['choices'][0]['text'].strip()
    except Exception as e:
        response = f'Sorry, I encountered the following error: \n {e}'

    return response

# **## Question Answering using RAG**

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input,top_k=20,max_tokens=256)

Llama.generate: prefix-match hit


'The management of sepsis in a critical care unit involves aggressive fluid resuscitation to maintain adequate tissue perfusion, prompt administration of appropriate antibiotics based on suspected causative organisms and sensitivity patterns, surgical excision or drainage of infected or necrotic tissues, supportive care, and sometimes intensive control of blood glucose and administration of corticosteroids and activated protein C. Early recognition and intervention are crucial to improve outcomes.'

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_2,top_k=20,max_tokens=256)

Llama.generate: prefix-match hit


"The common symptoms of appendicitis include epigastric or periumbilical pain followed by brief nausea, vomiting, and anorexia, which later shifts to the right lower quadrant. Pain increases with cough and motion. Classic signs are right lower quadalantar direct and rebound tenderness located at McBurney's point. However, these classic findings appear in less than 50% of patients, and symptoms may not be localized or tender. Appendicitis cannot be cured via medicine alone; instead, surgical removal through open or laparoscopic appendectomy is the standard treatment to prevent complications such as perforation, abscess formation, and peritonitis. Antibiotics are administered before surgery to reduce infection risk. If a large inflammatory mass involving the appendix, terminal ileum, and cecum is found, resection of the entire mass and ileocolostomy may be preferable. In late cases with pericolic abscesses, drainage by ultrasound-guided percutaneous catheter or open operation followed by

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_3,top_k=20,max_tokens=240)

Llama.generate: prefix-match hit


'Alopecia areata is a common cause of sudden patchy hair loss. It is an autoimmune disorder affecting genetically susceptible individuals exposed to unclear environmental triggers. The most effective treatments for alopecia areata include corticosteroids, either topical or oral, and immunomodulators such as minoxidil or anthralin. Corticosteroids help suppress the immune response that attacks the hair follicles, while minoxidil and anthralin promote hair regrowth. In severe cases, systemic corticosteroids may be used, but their long-term use is limited due to adverse effects. The diagnosis of alopecia areata is typically made based on clinical presentation, with confirmation through microscopic hair examination or scalp biopsy if necessary. Other causes of patchy hair loss include tinea capitis, trichotillomania, and certain scarring alopecias. Treatment for these conditions varies accordingly.'

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_4,top_k=20,max_tokens=240)

Llama.generate: prefix-match hit


'The initial treatment for traumatic brain injury (TBI) involves ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed for patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications of altered sensorium are important. Subsequently, many patients require rehabilitation. The causes of TBI include motor vehicle crashes, falls, assaults, and sports activities. The severity and consequences of injuries vary widely, with some individuals experiencing no gross structural damage while others have contusions or diffuse axonal injury. Concussion is defined as a transient and reversible posttraumatic alteration in mental status lasting from seconds to minutes and less than 6 hours. Treat

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_5,top_k=20,max_tokens=260)

Llama.generate: prefix-match hit


'A fractured leg requires prompt medical attention due to potential complications such as infection or compartment syndrome. The person should not bear weight on the affected limb and should seek medical care if they notice an odor from within the cast or a fever. For initial treatment, a splint can be used to immobilize the injury while allowing for ice application and movement. Prolonged immobilization may lead to complications such as stiffness, contractures, and muscle atrophy, so early mobilization is recommended for rapidly healing injuries. Fractures are typically treated with analgesics, immobilization, and sometimes surgery. In the emergency department, patients are evaluated for signs of ischemia or infection, and treatment may involve splinting, reduction (realignment of fracture fragments), and immobilization using a cast or surgical hardware. Patients should rest, apply ice, compress, and elevate the injured limb to minimize swelling and pain. Immobilization decreases pain

# **Fine-tuning**

Fine Tuning the hyperparameters (k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50)   can alter the response (output)

In [ ]:
user_input_ft01 = "What is the protocol for managing sepsis in a critical care unit?"
generate_rag_response(user_input_ft01,temperature=0.5,max_tokens=290)

Llama.generate: prefix-match hit


"The management of sepsis in a critical care unit involves aggressive fluid resuscitation to maintain adequate tissue perfusion, prompt administration of appropriate antibiotics based on suspected organisms and culture results, surgical excision or drainage of infected or necrotic tissues, supportive care, and sometimes intensive control of blood glucose levels. Corticosteroids and activated protein C may also be considered for certain patients. The goal is to improve outcomes and minimize suffering for dying patients while maintaining dignity. It's important to follow strict protocols for investigating alarms from monitoring devices and ensure prompt empiric therapy for suspected sepsis, as it can be lifesaving."

In [ ]:
user_input_ft02 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
generate_rag_response(user_input_ft02,temperature=0.5, max_tokens=280)

Llama.generate: prefix-match hit


"The classic symptoms of appendicitis include epigastric or periumbilical pain that shifts to the right lower quadrant with increasing intensity, loss of appetite, nausea and vomiting, and localized tenderness at McBurney's point. However, these findings are not present in all cases, especially among infants, children, elderly patients, and pregnant women. If untreated, appendicitis can lead to perforation, gangrene, abscess formation, and peritonitis, which require immediate surgical intervention. The treatment of choice for appendicitis is an appendectomy, either open or laparoscopic. In cases where surgery is impossible due to complications like perforation or the presence of a large inflammatory mass, antibiotics can improve survival rates but are not curative."

In [ ]:
user_input_ft03 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
generate_rag_response(user_input_ft03,temperature=0.5, max_tokens=256)

Llama.generate: prefix-match hit


'Alopecia Areata is a common cause of sudden patchy hair loss. It is an autoimmune disorder that affects genetically susceptible individuals exposed to unclear environmental triggers. Treatment options include topical or intralesional corticosteroids, topical minoxidil, topical anthralin, topical immunotherapy (diphencyprone or squaric acid dibutylester), or systemic corticosteroids in severe cases. Corticosteroids help to suppress the immune response and promote hair regrowth. However, once treatment is stopped, hair loss may return to previous levels. Other causes of patchy hair loss include tinea capitis, trichotillomania, lichen planopilaris, chronic cutaneous lupus lesions, and scarring alopecia due to various underlying disorders. Treatment for these conditions involves addressing the underlying cause. For example, antifungals are used for tinea capitis, behavior modification or medication for trichotillomania, and long-acting oral tetracyclines in combination with potent topical

In [ ]:
user_input_ft04 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
generate_rag_response(user_input_ft04,temperature=0.5, max_tokens=300)


Llama.generate: prefix-match hit


'The initial treatment for traumatic brain injury (TBI) involves ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure. Surgery may be needed in patients with more severe injuries to place monitors to track and treat intracranial pressure, decompress the brain if intracranial pressure is increased, or remove intracranial hematomas. In the first few days after the injury, maintaining adequate brain perfusion and oxygenation and preventing complications are important. Subsequently, many patients require rehabilitation. Treatment for mild injuries may involve discharge and observation, while moderate to severe injuries require optimization of ventilation, oxygenation, and brain perfusion, as well as treatment of complications such as increased intracranial pressure, seizures, and hematomas. Multiple non-cranial injuries, which are common with motor vehicle crashes and falls, often require simultaneous treatment. At the injury scene, a clear airway i

In [ ]:
user_input_ft04 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
generate_rag_response(user_input_ft04,temperature=0.5, max_tokens=252)

Llama.generate: prefix-match hit


'A fractured leg requires prompt medical attention. The person should not bear weight on the affected limb to prevent further damage. Immobilization using a splint or cast is necessary to stabilize the injury and allow healing. Ice should be applied intermittently during the first 24-48 hours to minimize swelling and pain. Rest, compression, and elevation (RICE) are essential for soft tissue injuries accompanying the fracture. In case of severe swelling, the cast may need to be bivalved or cut open. Prolonged immobilization can lead to complications such as stiffness, contractures, and muscle atrophy, especially in elderly individuals. Early mobilization is recommended for rapidly healing injuries to minimize these complications. Infection is a potential complication, so the person should be advised to keep their cast dry, never put objects inside it, inspect the edges daily, and apply lotion to any red or sore areas. If an odor emanates from within the cast or if a fever develops, med

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation.

We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message = ( """
You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score. """
)

In [ ]:
relevance_rater_system_message = (
    """ You are tasked with rating AI generated answers to questions posed by users.
You will be presented a question, context used by the AI system to generate the answer and an AI generated answer to the question.
In the input, the question will begin with ###Question, the context will begin with ###Context while the AI generated answer will begin with ###Answer.

Evaluation criteria:
The task is to judge the extent to which the metric is followed by the answer.
1 - The metric is not followed at all
2 - The metric is followed only to a limited extent
3 - The metric is followed to a good extent
4 - The metric is followed mostly
5 - The metric is followed completely

Metric:
The answer should be derived only from the information presented in the context

Instructions:
1. First write down the steps that are needed to evaluate the answer as per the metric.
2. Give a step-by-step explanation if the answer adheres to the metric considering the question and context as the input.
3. Next, evaluate the extent to which the metric is followed.
4. Use the previous information to rate the answer using the evaluaton criteria and assign a score."""
)

In [ ]:
user_message_template = """ ###Question
{question}

###Context
{context}

###Answer
{answer}"""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            echo=False
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

## Actionable Insights and Business Recommendations

**Query 1: What is the protocol for managing sepsis in a critical care unit?**

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What is the protocol for managing sepsis in a critical care unit?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the key information related to managing sepsis in a critical care unit from the context.
2. Compare the information identified in step 1 with the AI generated answer to determine if the answer is derived only from the context.

Explanation:
The context provides detailed information about the approach to caring for critically ill patients, including monitoring and testing, cardiac and pulmonary artery catheter monitoring, supportive care, and specific treatments for sepsis and septic shock. The AI generated answer summarizes the key components of managing sepsis in a critical care unit, which are consistent with the information provided in the context.

Evaluation:
The metric is followed mostly as the AI generated answer is primarily based on the information presented in the context, with some additional details and clarifications that do not deviate from the context.

Rating:
Based on the evaluation criteria, I would rate the answer a 4 (The m

Summary: The RAG system accurately retrieved and summarized sepsis protocols including early recognition, monitoring, fluid resuscitation, and infection control.
Insight: High groundedness (4/5) and relevance (5/5) confirm strong context alignment and medical reliability.

**Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?**

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to appendicitis and its treatment.
2. Check if the answer is derived only from the information presented in the context.
3. Compare each symptom, sign, and treatment mentioned in the answer with the corresponding information in the context.

The answer adheres to the metric as it mentions the common symptoms of appendicitis (epigastric or periumbilical pain that shifts to the right lower quadrant, nausea, vomiting, anorexia, and increased pain with coughing or movement) and signs (right lower quadrant direct and rebound tenderness at McBurney's point, Rovsing sign, psoas sign, or obturator sign), which are all present in the context. The answer also correctly states that appendicitis cannot be cured

 Steps to evaluate the answer:
1. Identify the information in the context related to appendicitis and its treatment.
2. Check if the answer is derived only from the information presented in the context.
3. Co

Summary: The model precisely described classic symptoms and confirmed that appendicitis requires surgical intervention (appendectomy).
Insight: Both groundedness and relevance rated 5/5, showing faithful retrieval from the medical context.

**Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?**

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the question and context.
2. Understand the user's query about effective treatments or solutions for sudden patchy hair loss (alopecia areata) and possible causes.
3. Read through the context provided, which discusses various treatments and causes of alopecia areata.
4. Analyze the AI-generated answer to ensure it is derived only from the information presented in the context.

Explanation:
The AI-generated answer mentions corticosteroids, topical anthralin, minoxidil, immunotherapy (diphencyprone or squaric acid dibutylester), and possible causes of alopecia areata. These treatments and causes are all discussed in the context provided. Therefore, the answer is derived solely from the information presented in the context.

Evaluation:
The metric is followed completely.

Rating:
Based on the evaluation criteria, I would rate the AI-generated answer as a 5 because it follows the metric completely.

 Steps to evaluate the answer:
1. Identify the q

Summary: The response correctly cited corticosteroids, minoxidil, and immunotherapy as key treatments and autoimmune causes.
Insight: Groundedness and relevance rated 4–5, indicating complete contextual alignment with minimal hallucination.

**Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?**

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the information in the context related to treatments for traumatic brain injury (TBI).
2. Compare the information from the context with the AI generated answer to determine if the answer is derived only from the context.

Explanation:
The context provides detailed information about the initial and subsequent treatments for TBI, including ensuring a reliable airway, maintaining adequate ventilation, oxygenation, and blood pressure, surgery for more severe injuries, rehabilitation, optimization of ventilation, oxygenation, and brain perfusion, treatment of complications such as increased intracranial pressure, seizures, and hematomas, and frequent monitoring of neurologic findings, blood pressure, pulse, and temperature.

The AI generated answer also mentions the initial treatments for TBI, including ensuring a reliable airway and maintaining adequate ventilation, oxygenation, and blood pressure. It also mentions surgery for more severe injuries

The system outlined accurate emergency and rehabilitation steps—airway control, oxygenation, surgery, and neurologic monitoring.
Insight: Perfect groundedness and relevance (5/5) demonstrate excellent medical consistency and retrieval fidelity.

**Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?**

In [ ]:

ground,rel = generate_ground_relevance_response(user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",max_tokens=370)

print(ground,end="\n\n")
print(rel)


Llama.generate: prefix-match hit
Llama.generate: prefix-match hit
Llama.generate: prefix-match hit


 Steps to evaluate the answer:
1. Identify the necessary precautions and treatment steps for a person with a fractured leg from the context.
2. Compare the identified steps with the AI generated answer to check if all the information is derived only from the context.

Explanation:
The AI generated answer adheres to the metric as it lists out the necessary precautions and treatment steps for a person with a fractured leg based on the information provided in the context. The answer mentions immobilization using a splint, application of ice packs, elevation of the injured limb, rest, seeking medical care for signs of infection or no improvement in symptoms, early mobilization for rapidly healing injuries, monitoring for complications during prolonged immobilization, and follow-up care including casting, physical therapy, or surgery. All these steps are directly derived from the context.

Rating:
Based on the above explanation, the metric is followed completely (rating 5) as all the inform

Summary: The answer comprehensively covered immobilization, ice application, elevation, early mobilization, and follow-up rehabilitation.
Insight: Both metrics rated 5/5, confirming precise grounding and medical completeness.

# **Actionable Insights and Business Recommendations**

The RAG pipeline (Mistral 7B + Chroma + thenlper/gte-large) consistently produced fact-based, context-grounded answers from the Merck Manual.
Implementing such a system in healthcare settings can reduce information overload, support rapid evidence-based decisions, and improve diagnostic consistency across practitioners.

1. Sepsis Management in Critical Care
	Model retrieved protocols for early recognition, fluid resuscitation, infection control, and monitoring.Groundedness = 4/5, Relevance = 5/5 — highly aligned and medically accurate.

2. Appendicitis – Symptoms & Surgical Treatment
 Correctly identified classic symptoms and surgical cure (appendectomy). Both metrics = 5/5 — perfectly grounded and contextually precise.

3. Sudden Patchy Hair Loss (Alopecia Areata)
 Accurately listed causes and treatments (corticosteroids, minoxidil, immunotherapy).
 Groundedness = 4–5/5 — fully supported by context, minimal hallucination.

4. Traumatic Brain Injury (TBI)
Outlined airway management, oxygenation, surgery, and rehabilitation steps. Both = 5/5 — exceptional retrieval fidelity and medical consistency.

5. Fractured Leg – Treatment & Recovery
 Comprehensive answer covering immobilization, elevation, mobilization, and follow-up care.
 Both = 5/5 — highly relevant and completely grounded.

# **Technical Recommendations**

Vector database scaling:
Indexing time rises with document size; may need to split large manuals into sections for faster vectorization and incremental updates.

Retriever tuning (k):
Answers may span multiple contexts—experiment with k = 3–6 to balance precision and completeness.

Chunk configuration: may be altering the chunk_size ≈ 500–800 and chunk_overlap ≈ 100–200 will benefit to preserve sentence continuity across fragments.

Token management:
Adjust max_tokens based on question complexity—larger budgets yield richer answers but may exceed model context limits.

Prompt & temperature control:
Refine prompts for focus; set temperature = 0 for factual answers or increase slightly for narrative summaries.

Continuous RAG optimization:
Iteratively tune retriever parameters (k, search_type, embeddings) per domain to improve performance.

Evaluation focus:
Prioritize groundedness (fact support) and relevance (context fit) in output validation to ensure reliability.

Feedback loop:
Maintain evaluation logs to retrain or re-rank embeddings, enhancing consistency across diverse query types.

# **Business Impact**

Implementing this RAG-based medical assistant can:

Reduce diagnostic information overload by surfacing verified knowledge from trusted manuals.

Improve decision speed and accuracy during critical care scenarios.

Standardize clinical reasoning by grounding recommendations in authoritative references.

<font size=6 color='blue'>Power Ahead</font>
___